**<div style='text-align: center; font-size: 48px'><u>Assignment 5: Fine-Tuning BERT for POS Tagging & Chunking </u></div>**

# **Task 0: Mounting and Loading Data**

In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [45]:
import os

def parse_conll_file(file_path):
    data = []
    if not os.path.exists(file_path):
        print(f">> Error: {file_path} not found.")
        return []
        
    with open(file_path, 'r', encoding='utf-8') as f:
        words, pos_tags, chunk_tags = [], [], []
        for line in f:
            line = line.strip()
            
            if not line or line.startswith("-DOCSTART-"):
                if words:
                    data.append({
                        "tokens": words, 
                        "pos_tags": pos_tags, 
                        "chunk_tags": chunk_tags
                    })
                    words, pos_tags, chunk_tags = [], [], []
                continue
            
            parts = line.split()
            if len(parts) >= 3:
                words.append(parts[0])
                pos_tags.append(parts[1])
                chunk_tags.append(parts[2])
    return data

train_data = parse_conll_file('train.txt')
valid_data = parse_conll_file('valid.txt')
test_data = parse_conll_file('test.txt')

if train_data:
    print(">> System: Data Loaded Successfully:")
    print(f"\tTrain sentences: {len(train_data)}")
    print(f"\tValidation sentences: {len(valid_data)}")
    print(f"\tTest sentences: {len(test_data)}")
else:
    print("⚠️ Data still not loading. Check file names in the sidebar!")

print("\n>> Sample Entry:")
print(f"\tTokens: {train_data[0]['tokens'][:5]}...")
print(f"\tPOS:    {train_data[0]['pos_tags'][:5]}...")

>> System: Data Loaded Successfully:
	Train sentences: 14041
	Validation sentences: 3250
	Test sentences: 3453

>> Sample Entry:
	Tokens: ['EU', 'rejects', 'German', 'call', 'to']...
	POS:    ['NNP', 'VBZ', 'JJ', 'NN', 'TO']...


# **Task 1: Data Selection**

In [46]:
unique_pos_tags = sorted(list(set(tag for sent in train_data for tag in sent['pos_tags'])))
unique_chunk_tags = sorted(list(set(tag for sent in train_data for tag in sent['chunk_tags'])))

print(">> Task 1: Label Identification")
print(f"\tTotal Unique POS tags: {len(unique_pos_tags)}")
print(f"\tTotal Unique Chunk tags: {len(unique_chunk_tags)}")

print(f"\n\tPOS tags (First 5): {unique_pos_tags[:5]}")
print(f"\tChunk tags (First 5): {unique_chunk_tags[:5]}")

>> Task 1: Label Identification
	Total Unique POS tags: 45
	Total Unique Chunk tags: 20

	POS tags (First 5): ['"', '$', "''", '(', ')']
	Chunk tags (First 5): ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST']


# **Task 2: Data Preprocessing**

In [47]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def fast_tokenize_and_align(data_list, label_key):
    # 1. Prepare raw lists
    tokens = [s["tokens"] for s in data_list]
    labels = [s[label_key] for s in data_list]
    
    # 2. Tokenize
    tokenized_inputs = tokenizer(tokens, truncation=True, is_split_into_words=True)
    
    all_labels = []
    for i in range(len(labels)):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) #
            elif word_idx != previous_word_idx:
                label_ids.append(label_l2i[labels[i][word_idx]]) #
            else:
                label_ids.append(-100) #
            previous_word_idx = word_idx
        all_labels.append(label_ids)
    
    # 3. Force format to standard Python lists to avoid ArrowTypeError
    return Dataset.from_dict({
        "input_ids": tokenized_inputs["input_ids"],
        "attention_mask": tokenized_inputs["attention_mask"],
        "labels": all_labels
    })

# Create Dataset objects
train_dataset = fast_tokenize_and_align(train_data, "pos_tags")
valid_dataset = fast_tokenize_and_align(valid_data, "pos_tags")

print(">> Preprocessing Verification:")

>> Preprocessing Verification:


# **Task 3: Model Setup**

In [48]:
from transformers import AutoModelForTokenClassification

pos_i2l = {i: tag for tag, i in label_l2i.items()}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(unique_pos),
    id2label=pos_i2l,
    label2id=label_l2i
)

print(f">> Task 3 Completed: Model '{model_checkpoint}' initialized")
print(f"\tTotal Labels: {model.config.num_labels}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


>> Task 3 Completed: Model 'distilbert-base-uncased' initialized
	Total Labels: 45


# **Task 4: Training**

In [49]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification


training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",           
    learning_rate=2e-5,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,
    num_train_epochs=3,              
    weight_decay=0.01,
    logging_dir='./logs',
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,     
    eval_dataset=valid_dataset,      
    data_collator=data_collator,
)

print(">> System: Starting Training...")
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


>> System: Starting Training...


Epoch,Training Loss,Validation Loss
1,0.756629,0.257803
2,0.203292,0.229055
3,0.162526,0.221577


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2634, training_loss=0.30239151279041054, metrics={'train_runtime': 285.1927, 'train_samples_per_second': 147.7, 'train_steps_per_second': 9.236, 'total_flos': 510454275094638.0, 'train_loss': 0.30239151279041054, 'epoch': 3.0})

# **Task 5: Evaluation**

In [53]:
!pip install evaluate
import evaluate
import numpy as np
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [pos_i2l[p] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [pos_i2l[l] for (p, l) in zip(prediction, label) if l!=-100] for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy'],
    }

trainer.compute_metrics = compute_metrics

eval_results = trainer.evaluate()

print("\n>> Task 5: Evaluation Report")
print(f"\tPrecision: {eval_results['eval_precision']:.4f}")
print(f"\tRecall: {eval_results['eval_recall']:.4f}")
print(f"\tF1-score: {eval_results['eval_f1']:.4f}")


>> Task 5: Evaluation Report
	Precision: 0.9204
	Recall: 0.9191
	F1-score: 0.9198


# **Task 6: Inference**

In [55]:
import torch

def predict_pos_tags(sentence):
    # 1. Tokenize the input
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True).to(model.device)

    # 2. Forward pass through the trained model
    with torch.no_grad():
        logits = model(**inputs).logits

    # 3. Get the highest probability token for each position
    predictions = torch.argmax(logits, dim=2)

    # 4. Map IDs back to labels and align with words
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    predicted_labels = [pos_i2l[p.item()] for p in predictions[0]]

    # Clean up: Ignore special tokens [CLS] and [SEP]
    result = []
    for token, label in zip(tokens, predicted_labels):
        if token not in tokenizer.all_special_tokens:
            result.append((token, label))

    return result

# 5. Example Test Case
test_sentence = "Rishi works at Innomatics Research Labs in Pune"
output = predict_pos_tags(test_sentence)

print(">> Task 6: Inference Output")
print(f"\tInput: {test_sentence}")
print(f"\tOutput: {output}")

>> Task 6: Inference Output
	Input: Rishi works at Innomatics Research Labs in Pune
	Output: [('ri', 'NNP'), ('##shi', 'NNP'), ('works', 'VBZ'), ('at', 'IN'), ('inn', 'NNP'), ('##oma', 'NNPS'), ('##tics', 'NNPS'), ('research', 'NNP'), ('labs', 'NNPS'), ('in', 'IN'), ('pune', 'NNP')]


# **Task 7: Comparison & Analysis**

Comparing **POS Tagging** with **Chunking**. In the NLP world, these are often ranked by difficulty.

**1. The Complexity Gap**
- **POS Tagging:** This is "point-wise" tagging. The model only needs to decide the grammatical category of a single token. Since words like "words" are almost always verbs, the context window required is small.
- **Chunking:** This is "segment-level" tagging. The model must not only identify the category but also the **boundaries** of phrase (Noun Phrases, Verb Phrases). It requires understanding the relationship $bet^{n}$ multiple words to decide where a "chunk" starts (`B-`) and where it continues(`I-`).


| Features | POS Tagging | Chunking |
|:---:|:---:|:---:|
| **Level** | Word-level (syntax) | Phrase-level (Semantics) |
| **Labels** | NNP, VBZ, IN, etc. | B-NP, I-NP, B-VP, etc |
| **Context Needed** | Low (Immediate neighbors) | High (Full phrase structure) |
| **Difficulty** | Easy | Medium |

**2. Why the Difference?**
- **Ambiguity:** A word's POS tag is often restricted by its spelling. However, a word's "Chunk" depends entirely on its role in a specific sentence.
- **Dependency:** In Chunking, a `B-NP` (Begin Noun Phrase) must be followed by a `I-NP` (Inside Noun Phrase) or a new chunk. This structural dependency makes it a harder "booster" for the model to learn compared to independent POS tags.